# How LLM Architecture Evolved, 2017–2026: A Short List of Substitutions

The full code of [How LLM Architecture Evolved, 2017–2026: A Short List of Substitutions](https://residual-stream.lvtan71.workers.dev/posts/llm-architecture-evolution/), run from [its folder](https://github.com/lvtan71/residual-stream-code/tree/main/posts/llm-architecture-evolution) in the
order the folder's README gives. Each script is followed by the output it
printed for the post, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/llm-architecture-evolution.ipynb)

**Hardware:** Runs on a CPU; Colab's free runtime is enough. About 1 min end to end.


The post's numbers come from an Apple M4 laptop. Other hardware and library
versions can change the last digits of timings and of some floating-point
results; the conclusions should not change.

In [ ]:
%pip install -q "torch==2.14.*" tokenizers

# Setup: fetch the companion repo and work in this post's folder.
import os, subprocess, sys
SLUG = "llm-architecture-evolution"
ROOT = os.path.abspath("..") if os.path.isdir("../posts") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
os.chdir(os.path.join(ROOT, "posts", SLUG))
print("working in", os.getcwd())

## `mlp_norm_swaps.py`

The tiny GPT with its SwiGLU MLP and RMSNorm, against a GELU MLP at 4D and against LayerNorm, one swap at a time, two seeds each, 4,000 steps on the dev slice. Runs `train.py` once per configuration.

In [ ]:
!{sys.executable} mlp_norm_swaps.py

The post printed:

```text
SwiGLU + RMSNorm (baseline) s0: 1000: 2.147 / 2000: 2.053 / 3000: 2.034 / 4000: 2.027  (61 passes)
GELU MLP, 4D wide         s0: 1000: 2.233 / 2000: 2.077 / 3000: 2.039 / 4000: 2.020  (61 passes)
LayerNorm instead of RMS  s0: 1000: 2.134 / 2000: 2.047 / 3000: 2.060 / 4000: 2.029  (61 passes)
SwiGLU + RMSNorm (baseline) s1: 1000: 2.140 / 2000: 2.030 / 3000: 2.023 / 4000: 2.006  (61 passes)
GELU MLP, 4D wide         s1: 1000: 2.246 / 2000: 2.074 / 3000: 2.022 / 4000: 2.009  (61 passes)
LayerNorm instead of RMS  s1: 1000: 2.127 / 2000: 2.051 / 3000: 2.028 / 4000: 2.017  (61 passes)
```

## `inference_memory.py`

The KV cache per generated token and the attention weights per layer for GPT-3 (MHA), Llama 2 70B and Llama 3 8B (GQA) and DeepSeek-V2 (MLA), and the parameters held against those used per token for two mixture-of-experts models.

In [ ]:
!{sys.executable} inference_memory.py

The post printed:

```text
GPT-3 175B (MHA)               KV cache    4608 KiB per token; attention weights per layer  604.0M (with full MHA:  604.0M)
Llama 2 70B (GQA, 8 groups)    KV cache     320 KiB per token; attention weights per layer  151.0M (with full MHA:  268.4M)
Llama 3 8B (GQA, 8 groups)     KV cache     128 KiB per token; attention weights per layer   41.9M (with full MHA:   67.1M)
DeepSeek-V2 (MLA)              KV cache      68 KiB per token; the same heads with MHA: 3,840 KiB, 57 times as much
Mixtral 8x7B                    47.0B parameters, 13.0B used per token (28%): per-token FLOPs of a 13B model, memory of a 47B one
DeepSeek-V2                    236.0B parameters, 21.0B used per token (9%): per-token FLOPs of a 21B model, memory of a 236B one
```

## `chinchilla.py`

Compute-optimal tokens per parameter from Chinchilla's scaling law as printed, with its unrounded constants, and as refit by Besiroglu et al.; then Llama 3 8B against the compute-optimal model at its training compute.

In [ ]:
!{sys.executable} chinchilla.py

The post printed:

```text
Hoffmann, as printed  tokens per parameter at C = 1e21 / 1e23 / 1e25 FLOPs:  50.1 /  78.2 / 122.1
Hoffmann, unrounded   tokens per parameter at C = 1e21 / 1e23 / 1e25 FLOPs:  34.0 /  50.7 /  75.7
Besiroglu refit       tokens per parameter at C = 1e21 / 1e23 / 1e25 FLOPs:  21.6 /  19.2 /  17.1

Llama 3 8B: 1,875 tokens per parameter, C = 7.2e+23 FLOPs
  Hoffmann, as printed  same compute: optimal  36B on  3.4T tokens, loss 1.923 vs the 8B's 1.949; same loss: a 26B model with 0.50x the 8B's training compute
  Hoffmann, unrounded   same compute: optimal  45B on  2.7T tokens, loss 1.911 vs the 8B's 1.944; same loss: a 29B model with 0.40x the 8B's training compute
  Besiroglu refit       same compute: optimal  81B on  1.5T tokens, loss 1.968 vs the 8B's 2.022; same loss: a 34B model with 0.18x the 8B's training compute
curve Hoffmann, as printed: 2:2.0266 4:1.9801 8:1.9485 16:1.9299 32:1.9228 64:1.9264 128:1.9405 256:1.9654
curve Hoffmann, unrounded: 2:2.0266 4:1.9777 8:1.9435 16:1.9221 32:1.9119 64:1.9121 128:1.9223 256:1.9426
curve Besiroglu refit: 2:2.1169 4:2.0622 8:2.0221 16:1.9941 32:1.9766 64:1.9688 128:1.9703 256:1.9813
```